# Notebook 2 - Analyse et figures

Dans ce notebook, notre groupe recharge le dataset final, refait l'analyse et genere les figures.

In [2]:
from pathlib import Path
import math
import textwrap
from typing import Dict, Iterable, List

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_DIR / "figures"
MERGED_PATH = PROCESSED_DIR / "ter_weather_current_regions_monthly.csv"

STRESS_COMPONENTS = {
    "heavy_rain_days": "Pluie forte",
    "very_heavy_rain_days": "Pluie tres forte",
    "snow_days": "Neige",
    "strong_wind_days": "Vent fort",
    "heat_days": "Chaleur",
    "frost_days": "Gel",
    "storm_days": "Orage",
}

COLORS = {
    "regularity": "#0B4F6C",
    "cancellation": "#E36414",
    "stress": "#9A031E",
    "accent": "#5F0F40",
    "calm": "#90BE6D",
    "middle": "#F9C74F",
    "shock": "#F94144",
    "grid": "#D0D7DE",
    "text": "#172B4D",
}


def weighted_average(values: pd.Series, weights: pd.Series) -> float:
    mask = values.notna() & weights.notna()
    if not mask.any():
        return math.nan
    return float(np.average(values[mask], weights=weights[mask]))


def shorten_comment(text: object, width: int = 110) -> str:
    if pd.isna(text) or not str(text).strip():
        return "Pas de commentaire SNCF disponible."
    clean = " ".join(str(text).split())
    return textwrap.shorten(clean, width=width, placeholder="...")


def add_value_labels(ax: plt.Axes, padding: float = 0.12) -> None:
    for patch in ax.patches:
        value = patch.get_height()
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            value + padding,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=10,
            color=COLORS["text"],
        )


def linear_fit(x: Iterable[float], y: Iterable[float]) -> dict:
    x_array = np.asarray(list(x), dtype=float)
    y_array = np.asarray(list(y), dtype=float)
    mask = np.isfinite(x_array) & np.isfinite(y_array)
    slope, intercept = np.polyfit(x_array[mask], y_array[mask], 1)
    correlation = float(np.corrcoef(x_array[mask], y_array[mask])[0, 1])
    return {"slope": float(slope), "intercept": float(intercept), "correlation": correlation}

In [3]:
if not MERGED_PATH.exists():
    raise FileNotFoundError(
        "Le fichier fusionne est absent. Lance d'abord le notebook 1 (API et nettoyage)."
    )

merged = pd.read_csv(MERGED_PATH, parse_dates=['date'])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(merged.shape)
merged.head()

(1738, 47)


,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,...,heat_days_stress,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label,stress_bucket,comment_clean
0,2013-01-01,Auvergne-Rhône-Alpes,"Auvergne, Rhône Alpes",37223.0,36511.0,712.0,3983.0,Conditions météos défavorables.,89.090959,1.912796,...,0.0,0.724224,0.724224,0.0,0.0,0.182470,frost_days,Gel,Mois intermédiaires,Conditions météos défavorables.
1,2013-01-01,Bourgogne-Franche-Comté,"Bourgogne, Franche Comté",14226.0,14076.0,150.0,1178.0,Un mois de janvier qui surpasse les six exerci...,91.631145,1.054407,...,0.0,0.355758,0.355758,0.0,0.0,0.259023,snow_days,Neige,Mois intermédiaires,Un mois de janvier qui surpasse les six exerci...
2,2013-01-01,Bretagne,Bretagne,8776.0,8631.0,145.0,554.0,Fortes chutes de neige ayant entrainé des pert...,93.581277,1.652233,...,0.0,0.404067,0.404067,0.0,0.0,0.339054,snow_days,Neige,Mois intermédiaires,Fortes chutes de neige ayant entrainé des pert...
3,2013-01-01,Centre-Val de Loire,Centre,9882.0,9687.0,195.0,812.0,"Trois incidents caténaires lourds, dont deux p...",91.617632,1.973285,...,0.0,0.529166,0.529166,0.0,0.0,0.422662,snow_days,Neige,Mois intermédiaires,"Trois incidents caténaires lourds, dont deux p..."
4,2013-01-01,Grand Est,"Alsace, Champagne Ardenne, Lorraine",NaN,NaN,NaN,NaN,Conditions météorologiques dégradées du 15 au ...,NaN,NaN,...,0.0,0.366391,0.366391,0.0,0.0,0.138331,snow_days,Neige,Mois intermédiaires,Conditions météorologiques dégradées du 15 au ...


## Indicateurs

In [4]:
def build_national_monthly(merged: pd.DataFrame) -> pd.DataFrame:
    rows: List[dict] = []
    for date, group in merged.groupby("date", sort=True):
        programmed = group["nombre_de_trains_programmes"].sum()
        circulated = group["nombre_de_trains_ayant_circule"].sum()
        cancelled = group["nombre_de_trains_annules"].sum()
        delayed = group["nombre_de_trains_en_retard_a_l_arrivee"].sum()
        rows.append(
            {
                "date": date,
                "regularity_pct": 100 * (circulated - delayed) / circulated,
                "cancellation_pct": 100 * cancelled / programmed,
                "weather_stress_score": weighted_average(group["weather_stress_score"], group["nombre_de_trains_programmes"]),
                "regions_covered": int(group["region_current"].nunique()),
            }
        )

    national = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    national["regularity_rolling_12m"] = national["regularity_pct"].rolling(12, min_periods=3).mean()
    national["stress_rolling_12m"] = national["weather_stress_score"].rolling(12, min_periods=3).mean()
    return national


def summarize_results(merged: pd.DataFrame) -> dict:
    relationship = linear_fit(merged["weather_stress_score"], merged["regularity_gap"])
    cancellation_relationship = linear_fit(merged["weather_stress_score"], merged["cancellation_gap"])

    region_sensitivity_rows: List[dict] = []
    for region, group in merged.groupby("region_current"):
        subset = group.dropna(subset=["weather_stress_score", "regularity_gap"])
        if len(subset) < 36:
            continue
        fit = linear_fit(subset["weather_stress_score"], subset["regularity_gap"])
        region_sensitivity_rows.append(
            {
                "region_current": region,
                "slope_regularite": fit["slope"],
                "correlation": fit["correlation"],
                "observations": len(subset),
            }
        )

    region_sensitivity = pd.DataFrame(region_sensitivity_rows).sort_values("slope_regularite").reset_index(drop=True)

    headline_metrics = pd.DataFrame(
        [
            {"indicateur": "Correlation stress meteo / Ecart de regularite", "valeur": relationship["correlation"]},
            {"indicateur": "Effet estime d'une unite de stress sur la regularite", "valeur": relationship["slope"]},
            {"indicateur": "Correlation stress meteo / Ecart d'annulation", "valeur": cancellation_relationship["correlation"]},
            {"indicateur": "Effet estime d'une unite de stress sur l'annulation", "valeur": cancellation_relationship["slope"]},
        ]
    )

    bucket_summary = (
        merged.groupby("stress_bucket", observed=False)
        .agg(
            observations=("date", "size"),
            regularity_gap_mean=("regularity_gap", "mean"),
            cancellation_gap_mean=("cancellation_gap", "mean"),
            regularity_pct_mean=("regularity_pct", "mean"),
            cancellation_pct_mean=("cancellation_pct", "mean"),
        )
        .reset_index()
    )

    top_weather_episodes = (
        merged[(merged["weather_stress_score"] >= merged["weather_stress_score"].quantile(0.85)) & (merged["regularity_gap"] < 0)]
        .sort_values(["regularity_gap", "weather_stress_score"], ascending=[True, False])
        .head(8)
        .copy()
    )
    top_weather_episodes["label"] = top_weather_episodes.apply(
        lambda row: f"{row['region_current']} - {row['date']:%Y-%m}", axis=1,
    )

    top_non_weather_episodes = (
        merged[merged["weather_stress_score"] <= merged["weather_stress_score"].quantile(0.20)]
        .sort_values("regularity_gap")
        .head(5)
        .copy()
    )

    return {
        "headline_metrics": headline_metrics,
        "bucket_summary": bucket_summary,
        "region_sensitivity": region_sensitivity,
        "top_weather_episodes": top_weather_episodes,
        "top_non_weather_episodes": top_non_weather_episodes,
    }


national = build_national_monthly(merged)
summary = summarize_results(merged)
summary["headline_metrics"]

,indicateur,valeur
0,Correlation stress meteo / Ecart de regularite,-0.182416
1,Effet estime d'une unite de stress sur la regu...,-1.467519
2,Correlation stress meteo / Ecart d'annulation,0.126809
3,Effet estime d'une unite de stress sur l'annul...,0.633162


In [5]:
summary['bucket_summary']

,stress_bucket,observations,regularity_gap_mean,cancellation_gap_mean,regularity_pct_mean,cancellation_pct_mean
0,Mois calmes,356,0.423239,-0.260078,92.002121,1.863673
1,Mois chocs météo,348,-0.570923,0.177974,90.720915,2.348594
2,Mois intermédiaires,1034,0.048558,0.028461,91.335499,2.221220


In [6]:
summary['region_sensitivity'][['region_current', 'slope_regularite', 'correlation', 'observations']]

,region_current,slope_regularite,correlation,observations
0,Nouvelle-Aquitaine,-2.529025,-0.306458,158
1,Normandie,-1.926476,-0.261305,158
2,Occitanie,-1.839339,-0.225526,158
3,Centre-Val de Loire,-1.422154,-0.204833,158
4,Provence-Alpes-Côte d'Azur,-1.335346,-0.084302,158
5,Hauts-de-France,-1.315873,-0.192641,156
6,Bretagne,-1.285891,-0.227768,158
7,Auvergne-Rhône-Alpes,-1.195030,-0.138631,158
8,Bourgogne-Franche-Comté,-1.102559,-0.174615,158
9,Grand Est,-0.982148,-0.239763,123


## Figures

In [7]:
def set_plot_style() -> None:
    plt.style.use("default")
    plt.rcParams.update(
        {
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.edgecolor": "#B6C2CF",
            "axes.labelcolor": COLORS["text"],
            "axes.titlesize": 14,
            "axes.titleweight": "bold",
            "axes.grid": True,
            "grid.color": COLORS["grid"],
            "grid.alpha": 0.6,
            "grid.linewidth": 0.8,
            "xtick.color": COLORS["text"],
            "ytick.color": COLORS["text"],
            "font.size": 11,
        }
    )

def plot_national_series(national: pd.DataFrame, path: Path) -> None:
    fig = plt.figure(figsize=(15, 9))
    grid = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.12)
    ax_top = fig.add_subplot(grid[0])
    ax_bottom = fig.add_subplot(grid[1], sharex=ax_top)

    ax_top.plot(
        national["date"],
        national["regularity_pct"],
        color=COLORS["regularity"],
        linewidth=2.4,
        label="Regularite TER nationale",
    )
    ax_top.plot(
        national["date"],
        national["regularity_rolling_12m"],
        color=COLORS["accent"],
        linewidth=1.6,
        linestyle="--",
        label="Tendance 12 mois",
    )
    ax_top.set_ylabel("Regularite (%)")
    ax_top.set_title("Regularite TER nationale et pression meteo au fil du temps")
    ax_top.legend(loc="lower left", frameon=False)

    ax_top_right = ax_top.twinx()
    ax_top_right.plot(
        national["date"],
        national["cancellation_pct"],
        color=COLORS["cancellation"],
        linewidth=1.4,
        alpha=0.8,
        label="Taux d'annulation",
    )
    ax_top_right.set_ylabel("Annulations (%)")
    ax_top_right.grid(False)

    ax_bottom.bar(
        national["date"],
        national["weather_stress_score"],
        width=24,
        color=COLORS["stress"],
        alpha=0.75,
    )
    ax_bottom.plot(
        national["date"],
        national["stress_rolling_12m"],
        color=COLORS["accent"],
        linewidth=1.5,
    )
    ax_bottom.set_ylabel("Score meteo")
    ax_bottom.set_xlabel("Date")
    ax_bottom.set_title("Score meteo national pondere par le volume de trains")

    top_shocks = national.nlargest(5, "weather_stress_score")
    for _, row in top_shocks.iterrows():
        ax_bottom.axvline(row["date"], color=COLORS["accent"], linewidth=0.8, alpha=0.35)
        ax_bottom.text(
            row["date"],
            row["weather_stress_score"] + 0.03,
            row["date"].strftime("%Y-%m"),
            rotation=90,
            ha="center",
            va="bottom",
            fontsize=9,
            color=COLORS["accent"],
        )

    locator = mdates.YearLocator(base=1)
    formatter = mdates.DateFormatter("%Y")
    ax_bottom.xaxis.set_major_locator(locator)
    ax_bottom.xaxis.set_major_formatter(formatter)
    plt.setp(ax_top.get_xticklabels(), visible=False)

    fig.suptitle(
        "Figure 1. La regularite baisse quand la pression meteo monte, avec des pics visibles lors des mois les plus exposes.",
        y=0.98,
        fontsize=15,
        fontweight="bold",
    )
    fig.text(
        0.01,
        0.01,
        "Sources : SNCF TER mensuel, Open-Meteo archive. Calculs maison sur regions harmonisees.",
        fontsize=9,
        color="#5E6C84",
    )
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [8]:
set_plot_style()
figure_1_path = FIGURES_DIR / '01_regularite_nationale_vs_stress_meteo.png'
plot_national_series(national, figure_1_path)
print(figure_1_path)

c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\01_regularite_nationale_vs_stress_meteo.png


![Figure 1](figures/01_regularite_nationale_vs_stress_meteo.png)

In [9]:
def plot_scatter_relationship(merged: pd.DataFrame, summary: dict, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(13, 8))
    sizes = 30 + 190 * (
        merged["nombre_de_trains_programmes"] / merged["nombre_de_trains_programmes"].quantile(0.95)
    ).clip(upper=1.5)
    scatter = ax.scatter(
        merged["weather_stress_score"],
        merged["regularity_gap"],
        s=sizes,
        c=merged["weather_stress_score"],
        cmap="YlOrRd",
        alpha=0.65,
        edgecolors="white",
        linewidths=0.4,
    )
    fit = linear_fit(merged["weather_stress_score"], merged["regularity_gap"])
    x_line = np.linspace(merged["weather_stress_score"].min(), merged["weather_stress_score"].max(), 200)
    y_line = fit["intercept"] + fit["slope"] * x_line
    ax.plot(x_line, y_line, color=COLORS["accent"], linewidth=2.4)
    ax.axhline(0, color="#7A869A", linestyle="--", linewidth=1)

    ax.set_xlabel("Score meteo mensuel")
    ax.set_ylabel("Ecart a la regularite habituelle (points)")
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Intensite du stress meteo")

    corr_value = summary["headline_metrics"].iloc[0, 1]
    slope_value = summary["headline_metrics"].iloc[1, 1]
    ax.text(
        0.02,
        0.98,
        f"Correlation = {corr_value:.2f}\nPente = {slope_value:.2f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "white", "edgecolor": COLORS["grid"]},
    )

    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [10]:
figure_2_path = FIGURES_DIR / '02_scatter_stress_vs_regularite_gap.png'
plot_scatter_relationship(merged, summary, figure_2_path)
print(figure_2_path)

c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\02_scatter_stress_vs_regularite_gap.png


![Figure 2](figures/02_scatter_stress_vs_regularite_gap.png)

In [11]:
def plot_region_sensitivity(region_sensitivity: pd.DataFrame, path: Path) -> None:
    ordered = region_sensitivity.sort_values("slope_regularite").copy()
    ordered["bar_color"] = np.where(
        ordered["slope_regularite"] < ordered["slope_regularite"].median(),
        COLORS["shock"],
        COLORS["regularity"],
    )

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(
        ordered["region_current"],
        ordered["slope_regularite"],
        color=ordered["bar_color"],
        alpha=0.9,
    )
    ax.axvline(0, color="#7A869A", linestyle="--", linewidth=1)
    ax.set_xlabel("Perte de regularite estimee pour une unite de stress meteo")

    for _, row in ordered.iterrows():
        ax.text(
            row["slope_regularite"] - 0.03,
            row["region_current"],
            f"n={int(row['observations'])}",
            ha="right",
            va="center",
            fontsize=9,
            color="white" if row["slope_regularite"] < -1.2 else COLORS["text"],
        )

    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [12]:
figure_3_path = FIGURES_DIR / '03_sensibilite_regionale_a_la_meteo.png'
plot_region_sensitivity(summary['region_sensitivity'], figure_3_path)
print(figure_3_path)

c:\Users\antoc\analyse-impact-meteo-regularite-sncf\figures\03_sensibilite_regionale_a_la_meteo.png


![Figure 3](figures/03_sensibilite_regionale_a_la_meteo.png)

In [ ]:
def plot_shock_vs_calm(bucket_summary: pd.DataFrame, path: Path) -> None:
    ordered = bucket_summary.set_index("stress_bucket").loc[
        ["Mois calmes", "Mois intermediaires", "Mois chocs meteo"]
    ].reset_index()
    palette = {
        "Mois calmes": COLORS["calm"],
        "Mois intermediaires": COLORS["middle"],
        "Mois chocs meteo": COLORS["shock"],
    }

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
    axes[0].bar(
        ordered["stress_bucket"],
        ordered["regularity_gap_mean"],
        color=[palette[label] for label in ordered["stress_bucket"]],
    )
    axes[0].axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    axes[0].set_title("Ecart moyen à la regularite habituelle")
    axes[0].set_ylabel("Points de regularite")
    add_value_labels(axes[0], padding=0.05)

    axes[1].bar(
        ordered["stress_bucket"],
        ordered["cancellation_gap_mean"],
        color=[palette[label] for label in ordered["stress_bucket"]],
    )
    axes[1].axhline(0, color="#7A869A", linestyle="--", linewidth=1)
    axes[1].set_title("Ecart moyen au taux d'annulation habituel")
    axes[1].set_ylabel("Points d'annulation")
    add_value_labels(axes[1], padding=0.05)

    for ax in axes:
        ax.tick_params(axis="x", rotation=15)

    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [15]:
figure_4_path = FIGURES_DIR / '04_mois_calmes_vs_mois_chocs.png'
plot_shock_vs_calm(summary['bucket_summary'], figure_4_path)
print(figure_4_path)

KeyError: "['Mois intermediaires', 'Mois chocs meteo'] not in index"

![Figure 4](figures/04_mois_calmes_vs_mois_chocs.png)

In [16]:
def plot_top_weather_episodes(top_weather_episodes: pd.DataFrame, path: Path) -> None:
    ordered = top_weather_episodes.sort_values("regularity_gap").copy()

    fig = plt.figure(figsize=(15, 8))
    grid = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.4], wspace=0.08)
    ax_bar = fig.add_subplot(grid[0])
    ax_text = fig.add_subplot(grid[1])

    ax_bar.barh(
        ordered["label"],
        ordered["regularity_gap"],
        color=COLORS["shock"],
        alpha=0.88,
    )
    ax_bar.axvline(0, color="#7A869A", linestyle="--", linewidth=1)
    ax_bar.set_xlabel("Perte vs niveau habituel (points)")
    ax_bar.set_title("Episodes meteo les plus penalissants")

    for _, row in ordered.iterrows():
        ax_bar.text(
            row["regularity_gap"] - 0.05,
            row["label"],
            f"{row['regularity_gap']:.1f} pts",
            ha="right",
            va="center",
            color="white",
            fontsize=10,
        )

    ax_text.axis("off")
    ax_text.set_title("Lecture rapide pour l'oral", loc="left")
    total = max(len(ordered), 1)
    for idx, row in enumerate(ordered.itertuples(index=False), start=1):
        y = 1 - (idx - 0.5) / total
        ax_text.text(
            0.0,
            y,
            (
                f"{row.label}\n"
                f"Stress = {row.weather_stress_score:.2f} | Driver = {row.dominant_weather_label}\n"
                f"Regularite = {row.regularity_pct:.1f}% | Annulations = {row.cancellation_pct:.1f}%\n"
                f"{row.comment_clean}"
            ),
            va="top",
            fontsize=10,
            color=COLORS["text"],
        )

    fig.suptitle(
        "Figure 5. Les episodes les plus explicables par la meteo sont parlants et faciles a commenter a l'oral.",
        y=0.98,
        fontsize=15,
        fontweight="bold",
    )
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)

In [ ]:
figure_5_path = FIGURES_DIR / '05_episodes_meteo_marquant.png'
plot_top_weather_episodes(summary['top_weather_episodes'], figure_5_path)
print(figure_5_path)

![Figure 5](figures/05_episodes_meteo_marquant.png)

## Tables

In [17]:
summary['top_weather_episodes']

,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,...,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label,stress_bucket,comment_clean,label
252,2014-11-01,Provence-Alpes-Côte d'Azur,Provence Alpes Côte d'Azur,12950.0,12179.0,771.0,3122.0,La production est confrontée à de fortes intem...,74.365711,5.953668,...,0.000000,0.000000,0.0,0.0,0.590003,heavy_rain_days,Pluie forte,Mois chocs météo,La production est confrontée à de fortes intem...,Provence-Alpes-Côte d'Azur - 2014-11
1437,2023-11-01,Nouvelle-Aquitaine,Nouvelle Aquitaine,15518.0,14682.0,836.0,3214.0,NaN,78.109249,5.387292,...,0.162221,0.162221,0.0,0.0,1.016357,very_heavy_rain_days,Pluie très forte,Mois chocs météo,Pas de commentaire SNCF disponible.,Nouvelle-Aquitaine - 2023-11
920,2019-12-01,Nouvelle-Aquitaine,Aquitaine,3888.0,3635.0,253.0,752.0,NaN,79.312242,6.507202,...,-0.850104,0.000000,0.0,0.0,0.861540,strong_wind_days,Vent fort,Mois chocs météo,Pas de commentaire SNCF disponible.,Nouvelle-Aquitaine - 2019-12
142,2014-01-01,Provence-Alpes-Côte d'Azur,Provence Alpes Côte d'Azur,14703.0,13336.0,1367.0,2652.0,NaN,80.113977,9.297422,...,-0.595683,0.000000,0.0,0.0,0.937389,heavy_rain_days,Pluie forte,Mois chocs météo,Pas de commentaire SNCF disponible.,Provence-Alpes-Côte d'Azur - 2014-01
1735,2026-02-01,Occitanie,Occitanie,13501.0,12365.0,1480.0,2263.0,NaN,81.698342,10.962151,...,-0.853084,0.000000,0.0,0.0,0.807809,very_heavy_rain_days,Pluie très forte,Mois chocs météo,Pas de commentaire SNCF disponible.,Occitanie - 2026-02
24,2013-03-01,Bretagne,Bretagne,8363.0,8208.0,155.0,924.0,Perte de régularité supérieure à 6 points en g...,88.742690,1.853402,...,2.846050,2.846050,0.0,0.0,1.154438,snow_days,Neige,Mois chocs météo,Perte de régularité supérieure à 6 points en g...,Bretagne - 2013-03
674,2018-02-01,Centre-Val de Loire,Centre,9898.0,9622.0,276.0,1439.0,Un mois de février catastrophique en matière d...,85.044689,2.788442,...,2.002191,2.002191,0.0,0.0,0.662763,snow_days,Neige,Mois chocs météo,Un mois de février catastrophique en matière d...,Centre-Val de Loire - 2018-02
941,2020-02-01,Normandie,Normandie,10373.0,10000.0,373.0,1279.0,NaN,87.210000,3.595874,...,-1.089992,0.000000,0.0,0.0,1.118639,strong_wind_days,Vent fort,Mois chocs météo,Pas de commentaire SNCF disponible.,Normandie - 2020-02


In [18]:
summary['top_non_weather_episodes']

,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,...,heat_days_stress,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label,stress_bucket,comment_clean
109,2013-10-01,Provence-Alpes-Côte d'Azur,Provence Alpes Côte d'Azur,11011.0,10306.0,705.0,2382.0,Mouvement social interprofessionnel.,76.887250,6.402688,...,0.0,0.000000,0.0,0.0,0.0,0.0,heavy_rain_days,Pluie forte,Mois calmes,Mouvement social interprofessionnel.
930,2020-01-01,Normandie,Normandie,8356.0,8102.0,254.0,1297.0,NaN,83.991607,3.039732,...,0.0,-0.834990,0.0,0.0,0.0,0.0,heavy_rain_days,Pluie forte,Mois calmes,Pas de commentaire SNCF disponible.
563,2017-04-01,Bretagne,Bretagne,7378.0,7303.0,75.0,874.0,NaN,88.032315,1.016536,...,0.0,-0.613572,0.0,0.0,0.0,0.0,heavy_rain_days,Pluie forte,Mois calmes,Pas de commentaire SNCF disponible.
175,2014-04-01,Provence-Alpes-Côte d'Azur,Provence Alpes Côte d'Azur,13459.0,12957.0,502.0,2413.0,La circulation des trains est rétablie entre G...,81.376862,3.729846,...,0.0,0.000000,0.0,0.0,0.0,0.0,heavy_rain_days,Pluie forte,Mois calmes,La circulation des trains est rétablie entre G...
615,2017-08-01,Provence-Alpes-Côte d'Azur,Provence Alpes Côte d'Azur,14967.0,14450.0,517.0,2639.0,"Dans la continuité du mois de juillet, le mois...",81.737024,3.454266,...,0.0,0.000000,0.0,0.0,0.0,0.0,heavy_rain_days,Pluie forte,Mois calmes,"Dans la continuité du mois de juillet, le mois..."


Ce qu'on retient pour notre groupe : la meteo compte, mais elle n'explique pas tout.